In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

MINIO_ACCESS = "slavakoder"
MINIO_SECRET = "slavakoder"
DB_PASS = "airflow"

spark = SparkSession.builder \
    .appName('cleandata') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .config('spark.shuffle.partitions', '8') \
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.demo.type", "jdbc") \
    .config("spark.sql.catalog.demo.uri", "jdbc:postgresql://postgres:5432/airflow") \
    .config("spark.sql.catalog.demo.jdbc.user", "airflow") \
    .config("spark.sql.catalog.demo.jdbc.password", DB_PASS) \
    .config("spark.sql.catalog.demo.warehouse", "s3a://raw-bronze/warehouse") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.postgresql:postgresql:42.6.0") \
    .getOrCreate()
print('запускаемся')

spark.sparkContext.setLogLevel('WARN')

raw_data = 's3a://raw-bronze/landing/p2p_transfers/*.csv'

ddl_schema = "tx_id STRING, sender_id STRING, receiver_id STRING, amount STRING, currency STRING, status STRING, timestamp LONG"

df = spark.read.csv(raw_data, header=True, schema=ddl_schema)

запускаемся


In [14]:
df = (df
    .withColumn('timestamp', F.from_unixtime(F.col('timestamp')).cast('timestamp'))
    .withColumn('sender_id', F.regexp_replace(F.col('sender_id'), r'\s+', ''))
    .withColumn('receiver_id', F.regexp_replace(F.col('receiver_id'), r'\s+', ''))
)

AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "from_unixtime(timestamp, yyyy-MM-dd HH:mm:ss)" due to data type mismatch: Parameter 1 requires the "BIGINT" type, however "timestamp" has the type "TIMESTAMP".;
'Project [tx_id#0, sender_id#85, receiver_id#93, amount#3, currency#4, status#5, cast(from_unixtime(timestamp#77, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC)) as timestamp) AS timestamp#275]
+- Project [tx_id#0, sender_id#85, receiver_id#93, amount#3, currency#4, status#5, timestamp#77]
   +- Project [tx_id#0, sender_id#85, receiver_id#93, amount#3, currency#4, status#5, timestamp#77]
      +- Project [tx_id#0, sender_id#85, regexp_replace(receiver_id#31, \s+, , 1) AS receiver_id#93, amount#3, currency#4, status#5, timestamp#77]
         +- Project [tx_id#0, regexp_replace(sender_id#23, \s+, , 1) AS sender_id#85, receiver_id#31, amount#3, currency#4, status#5, timestamp#77]
            +- Project [tx_id#0, sender_id#23, receiver_id#31, amount#3, currency#4, sta

In [13]:
df = df.fillna('1970-01-01 00:00:00', subset=['timestamp'])
df.show(20, truncate=False)

+--------------------------------+---------+-----------+-------+--------+--------+-------------------+
|tx_id                           |sender_id|receiver_id|amount |currency|status  |timestamp          |
+--------------------------------+---------+-----------+-------+--------+--------+-------------------+
|2ab1a9d4d25e4a57b14bdc0d7aec8718|USR_1230 |USR_10667  |2002,27|RUB     |SUCCESS |2026-04-26 15:12:13|
|0611ae28fe9f4a55b8597ca297bf4bbe|USR_26909|USR_41686  |3472,57|EUR     |PENDING |NULL               |
|845543bde4e74765aed5716406bde8b4|USR_20263|USR_33520  |1142,09|USD     |PENDING |2026-05-19 01:06:05|
|9e84d8998a844f81877a06ce3d84877a|USR_36964|USR_42362  |3362,77|USD     |SUCCESS |2026-04-27 08:27:39|
|0c30a4bd7bad4c0780945de67baaf5ea|USR_35099|USR_30101  |1720,66|RUB     |SUCCESS |2026-04-25 14:51:36|
|80db6361883a4911a9c27ab34a5f94ac|USR_8676 |USR_33634  |4926,43|USDT    |N/A     |2026-05-21 08:11:26|
|ecadb364f54e46ffa00a361fe9351331|USR_42649|USR_4122   |530,64 |RUB     |

In [9]:
df.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- sender_id: string (nullable = true)
 |-- receiver_id: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)

